In [0]:
%sql
  -- Bronze Table
CREATE TABLE IF NOT EXISTS banking_bronze.dim_customer_bronze (
   CUST_ID STRING, NAME STRING, Risk_Score INT, KYC_Status STRING, updated_at TIMESTAMP);

   -- Silver Table
CREATE TABLE IF NOT EXISTS banking_silver.dim_customer_silver (
    CUST_ID STRING, NAME_MASKED STRING, Risk_Score INT, KYC_Status STRING, updated_at TIMESTAMP
) USING DELTA;

-- Gold SCD1 Table (CDF Enabled)
CREATE TABLE IF NOT EXISTS banking_gold.dim_customer_gold_scd1 (
    CUST_ID STRING, NAME_MASKED STRING, Risk_Score INT, KYC_Status STRING, updated_at TIMESTAMP
) USING DELTA TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Gold SCD2 Table (Includes DLT-style history columns)
CREATE TABLE IF NOT EXISTS banking_gold.dim_customer_gold_scd2 (
    CUST_ID STRING, NAME_MASKED STRING, Risk_Score INT, KYC_Status STRING, updated_at TIMESTAMP,
    start_at TIMESTAMP, end_at TIMESTAMP
) USING DELTA;



In [0]:
%sql
select * from banking_bronze.dim_customer_bronze limit 10;

In [0]:
%sql
--1. CDC (History load for the first time alone) applying Watermarking Feature using Foreign Catalog
INSERT INTO banking_bronze.dim_customer_bronze
SELECT 
    CUST_ID,
    NAME,
    Risk_Score, 
    KYC_Status,
    updated_at 
FROM banking_foreign_catalog.bank_raji_schema.dim_customer
WHERE updated_at > (SELECT COALESCE(MAX(updated_at), '1900-01-01') 
    FROM banking_bronze.dim_customer_bronze);
    --This is checkpointing or watermarking feature to achieve incremental load/CDC

In [0]:
%python
from pyspark.sql.functions import col, upper, trim, sha2, row_number,concat_ws
from pyspark.sql.window import Window

# ====================================================================
# 1. READ BRONZE & LOAD CURATED SILVER (DEDUPLICATED BATCH)
# ====================================================================

raw_df = spark.read.table("banking_bronze.dim_customer_bronze")

# Define Window to identify the latest record per device in the incoming batch
window_spec = Window.partitionBy("CUST_ID").orderBy(col("updated_at").desc())

# Apply filters, transformations, and deduplication
silver_df = (
    raw_df
    .filter(col("CUST_ID").isNotNull()) 
    .withColumn("CUST_ID", col("CUST_ID").cast("string"))
    # Mask customer name
    .withColumn("NAME_MASKED", sha2(col("NAME"), 256))
    # Standardize KYC
    .withColumn("KYC_Status", upper(trim(col("KYC_Status"))))

    # Deduplication Logic
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn", "NAME")
)

# Save the deduplicated batch to Silver
silver_df.write.format("delta").mode("overwrite").saveAsTable("banking_silver.dim_customer_silver")

# ====================================================================
# 2. SHIFTING TO SQL MERGE FOR GOLD LAYER (SIMPLE UPSERT)
# ====================================================================

silver_df.createOrReplaceTempView("dedup_latest_silver")

# ====================================================================
# 3. GOLD LAYER: SCD TYPE 1 (Hard Deletes + Hash Comparison)
# ====================================================================
spark.sql("""
    MERGE INTO banking_gold.dim_customer_gold_scd1 t
    USING dedup_latest_silver s
    ON t.CUST_ID = s.CUST_ID
    
    -- Hard delete if inactive (based on KYC_Status or rule)
    WHEN MATCHED AND UPPER(s.KYC_Status) = 'INACTIVE' THEN 
        DELETE -- Hard delete

    -- Update if data changed    
    WHEN MATCHED AND s.updated_at > t.updated_at 
                 AND sha2(concat_ws('||',
                    s.NAME_MASKED,
                    s.Risk_Score,
                    s.KYC_Status
    ), 256) <> sha2(concat_ws('||',
        t.NAME_MASKED,
        t.Risk_Score,
        t.KYC_Status
    ), 256) THEN
        UPDATE SET 
        t.NAME_MASKED = s.NAME_MASKED,
        t.Risk_Score = s.Risk_Score,
        t.KYC_Status = s.KYC_Status,
        t.updated_at = s.updated_at
        
    WHEN NOT MATCHED AND s.KYC_Status != 'INACTIVE' THEN 
        INSERT (
    CUST_ID,
    NAME_MASKED,
    Risk_Score,
    KYC_Status,
    updated_at
)
VALUES (
    s.CUST_ID,
    s.NAME_MASKED,
    s.Risk_Score,
    s.KYC_Status,
    s.updated_at
)
""")

# ====================================================================
# 4. GOLD LAYER: SCD TYPE 2 (Soft Deletes + Hash Comparison)
# ====================================================================

staged_updates = spark.sql("""
    SELECT s.CUST_ID AS merge_key, s.* 
    FROM dedup_latest_silver s
    
    UNION ALL
    
    SELECT NULL AS merge_key, s.*
    FROM dedup_latest_silver s
    JOIN banking_gold.dim_customer_gold_scd2 t
      ON s.CUST_ID = t.CUST_ID
    WHERE t.end_at IS NULL 
      AND (
        s.NAME_MASKED <> t.NAME_MASKED OR
        s.Risk_Score <> t.Risk_Score OR
        s.KYC_Status <> t.KYC_Status
      )
""")

staged_updates.createOrReplaceTempView("staged_updates")

spark.sql("""
    MERGE INTO banking_gold.dim_customer_gold_scd2 t
    USING staged_updates s
    ON t.CUST_ID = s.merge_key AND t.end_at IS NULL
    
    WHEN MATCHED AND (
        s.NAME_MASKED <> t.NAME_MASKED OR
        s.Risk_Score <> t.Risk_Score OR
        s.KYC_Status <> t.KYC_Status
    ) THEN
    UPDATE SET
        t.end_at = s.updated_at
    
    WHEN NOT MATCHED THEN
    INSERT (
        CUST_ID,
        NAME_MASKED,
        Risk_Score,
        KYC_Status,
        updated_at,
        start_at,
        end_at
    )
    VALUES (
        s.CUST_ID,
        s.NAME_MASKED,
        s.Risk_Score,
        s.KYC_Status,
        s.updated_at,
        s.updated_at,
        NULL
    )
""")